In [0]:
bronze_df = spark.read.table("genai.bronze.docs_raw")
display(bronze_df)

In [0]:
from databricks.sdk import WorkspaceClient
for e in WorkspaceClient().serving_endpoints.list():
    print(e.name, e.state.ready if e.state else None)

In [0]:
%sql
WITH parsed_docs AS(
SELECT path, ai_parse_document(content,map('version','2.0')) as parsed_content
from genai.bronze.docs_raw)
SELECT path,
parsed_content,ai_query('databricks-gpt-oss-120b',
CONCAT('Given a JSON object representing a parsed documnet (with pages, elements, and metadata), convert the content into clean, readable markdown.Use "==page==" to seperate each page without adding page number.Preserve important structure such as pages, headers, tables, and captions.  
JSON:',CAST(parsed_content AS STRING)) as p
) from parsed_docs


In [0]:
from pyspark.sql.functions import expr

endpoint = 'databricks-gpt-oss-120b'

prompt_prefix = """
Given a JSON object representing a parsed documnet (with pages, elements, and metadata), convert the content into clean, readable markdown.Use "==page==" to seperate each page without adding page number.Preserve important structure such as pages, headers, tables, and captions.  
JSON:
"""

parsed_df = (
    bronze_df
    .withColumn(
        "parsed_content",
        expr("ai_parse_document(content, map('version', '2.0'))")
    )
    .withColumn(
        "plain_text",
        expr(f"""ai_query(
            '{endpoint}',
            request => concat(
                '{prompt_prefix.replace("'", "''")}',
                CAST(parsed_content AS STRING)
            )
        )""")
    )
    .select("path", "parsed_content", "plain_text")
)
display(parsed_df)

In [0]:
parsed_df.write.mode('overwrite').saveAsTable('genai.silver.docs_parsed')

In [0]:
%sql
select * from genai.silver.docs_parsed